### Housing Prices Competition for Kaggle Learn Users

    Description:
        Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or 
        the proximity to an east-west railroad. But this playground competition's dataset proves that much more influences price 
        negotiations than the number of bedrooms or a white-picket fence.

        79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this competition challenges 
        you to predict the final price of each home.
        
    Goal:
        It is your job to predict the sales price for each house. 
        For each Id in the test set, you must predict the value of the SalePrice variable.
        
    Metric:
            Submissions are evaluated on Root-Mean-Squared-Error (RMSE) between the logarithm of the predicted value and 
            the logarithm of the observed sales price. (Taking logs means that errors in predicting expensive houses 
            and cheap houses will affect the result equally.)

#### EDA of train.csv and test.csv file
    1. Data Ingestion
    2. Data Cleanup
    3. Data Statistics
    4. Data Visualization
    5. Problem Solving requirements
    

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from scipy.stats import f_oneway
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
import numpy as np

def check_duplicates(df: pd.DataFrame) -> bool:
    duplicate_exists = False
    # Check Dupliates in train_df dataset
    duplicate_count = df.duplicated().sum()    
    if duplicate_count > 0:
        print("\033[91mDuplicates exists. Need to remove duplicates\033[0m")
        duplicate_exists = True
    else:
        print("\033[92mDuplicates does not exist. Go to next step for checking nulls\033[0m")
        duplicate_exists = False
    return duplicate_exists

def check_null_values(df: pd.DataFrame) -> bool:
    null_values_exist = False
    # Check Nulls for df dataset
    missing_vals = df.isna().sum().sum()
    missing_values = df.isna().sum()
    if  missing_vals:
        print(f"\033[91mNull Values exists for DataFrame. Need to replace nulls appropriately\033[0m")
        null_values_exist = True
    else:
        print(f"\033[92mNull Values does not exist for DataFrame.\033[0m")
        null_values_exist = False
    return null_values_exist

def get_null_cols_list(df: pd.DataFrame):
    return list(df.columns[df.isnull().any()])

def get_fill_values(df: pd.DataFrame) -> dict:
    fill_values = {}
    cat_cols = df.select_dtypes(include='object').columns
    
    for col in df.columns:
        if col in cat_cols:
            fill_values[col] = df[col].mode()[0]
        else:
            skew_val = df[col].skew()
            if abs(skew_val) > 0.5:
                fill_values[col] = df[col].median()
            else:
                fill_values[col] = df[col].mean()
    return fill_values

def apply_fill_values(df: pd.DataFrame, fill_values: dict):
    for col, value in fill_values.items():
        if col in df.columns:
            df[col] = df[col].fillna(value)
    print("Missing values replaced using training data statistics.")


# Get Path
train_path = r"D:\Career-Related\projects\learning\kaggle-dataset-examples\home-data-for-ml-course\train.csv"
train_df = pd.read_csv(train_path, keep_default_na=False, na_values=["", "NA"])
# print("Train CSV Datatypes, Shape")
# print(train_df.shape, train_df.columns)
test_path = r"D:\Career-Related\projects\learning\kaggle-dataset-examples\home-data-for-ml-course\test.csv"
test_df = pd.read_csv(test_path, keep_default_na=False, na_values=["", "NA"])
# print("Test CSV Datatypes, Shape")
# print(test_df.shape)

# Check Duplicates
print("=========== Check Duplicates in DataFrame =========== Start")
train_data_duplicate_exists = check_duplicates(train_df)
print("Train Data Duplicate Exists: "+str(train_data_duplicate_exists))
test_data_duplicate_exists = check_duplicates(test_df)
print("Test Data Duplicate Exists: "+str(test_data_duplicate_exists))
print("=========== Check Duplicates in DataFrame =========== End")

print("=========== Check Nulls in DataFrame =========== Start")
train_data_null_values_exists = check_null_values(train_df)
print("Train Data Null Values Exists: "+str(train_data_null_values_exists))
test_data_null_values_exists = check_null_values(test_df)
print("Test Data Null Values Exists: "+str(test_data_null_values_exists))
print("=========== Check Nulls in DataFrame =========== End")

print("=========== Get Nulls Columns in DataFrame =========== Start")
train_data_null_cols = get_null_cols_list(train_df)
print("Train Data Null Values Exists: "+str(train_data_null_cols))
test_data_null_cols = get_null_cols_list(test_df)
print("Test Data Null Values Exists: "+str(test_data_null_cols))
print("=========== Get Nulls Columns in DataFrame =========== End")

print("=========== Update Nulls Columns in DataFrame =========== Start")
# On train data
fill_values = get_fill_values(train_df)
apply_fill_values(train_df, fill_values)

# On test data — use same values
apply_fill_values(test_df, fill_values)
print("=========== Update Nulls Columns in DataFrame =========== End")

print("=========== Check Nulls in DataFrame =========== Start")
train_data_null_values_exists = check_null_values(train_df)
print("Train Data Null Values Exists: "+str(train_data_null_values_exists))
test_data_null_values_exists = check_null_values(test_df)
print("Test Data Null Values Exists: "+str(test_data_null_values_exists))
print("=========== Check Nulls in DataFrame =========== End")
# Problem Statement: Identify features from train data which affect SalePrice
# Approach:
#       1. Identify all numeric columns in train data  
#       2. Find correlation of all the columns against SalePrice and identify the list with highest value > 0.5 and < 0.5
#       3. Store the columns in a numerical features list
#       4. Identify categorical columns in train data
#       5. Perform ANOVA test on categorical columns of train data and identify columns with needs encoding
#       6. Drop categorical columns from train and test data which have p-val > 0.05
#       7. Seggregate categorical columns into ordinal and nominal for ordinal / onehot encoding
#       8. Apply ordinal / onehot on categorical columns on train and test data
#       9. Create Features list by excluding SalePrice, Id columns and target as SalePrice
#       10. Find RMSE using LinearRegression using cross val score
#       11. Perform predictions on log transform and save the values to submission.csv file

# Step1 - Identify all numeric columns in train data
correlation_cols = train_df.corr(numeric_only=True).corr()['SalePrice'].sort_values(ascending=False)
# Step 2 - Find correlation of all the columns against SalePrice and identify the list with highest value > 0.5 and < 0.5
strong_corr_cols = correlation_cols[abs(correlation_cols) > 0.5].drop('SalePrice')
# Step 3 - Store the columns in a features list
numerical_features_list = strong_corr_cols.index.tolist()
# Step 4 - Identify categorical columns in train data
categorical_cols = train_df.select_dtypes(include='object').columns
print(categorical_cols)

# Step 5 - ANOVA
categorical_features_list = []
for col in categorical_cols:
    groups = [group['SalePrice'].values for _, group in train_df.groupby(col)]
    f_stat, p_val = f_oneway(*groups)
    if p_val < 0.05:
        categorical_features_list.append(col)
    else:
        print(f"{col} is not candidate for encoding. Adding to categorical_features_list")
print(len(categorical_features_list), len(categorical_cols))

# Step 6 - Drop unused categorical columns not selected by ANOVA
excluded_cats = list(set(categorical_cols) - set(categorical_features_list))
train_df.drop(columns=excluded_cats, inplace=True)
test_df.drop(columns=excluded_cats, inplace=True)

# Step-7 - Segregate Categorical columns into Nominal and Ordinal columns
nominal_features = [
    'MSSubClass',     # Technically numeric, but categorical
    'MSZoning',
    'Street',
    'Alley',
    'LotConfig',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'MiscFeature',
    'SaleType',
    'SaleCondition'
]

ordinal_features_list = []
nominal_features_list = []
for col in categorical_features_list:
    if col in nominal_features:
        nominal_features_list.append(col)
    else:
        ordinal_features_list.append(col)
print(f"Nominal Features Size: {len(nominal_features_list)}, Ordinal Features Size: {len(ordinal_features_list)}")
print(f"Nominal Features List: {nominal_features_list}, Ordinal Features List: {ordinal_features_list}")

# Ordinal Features with logical orderings
ordinal_features = {
    'LotShape': {'IR3': 1, 'IR2': 2, 'IR1': 3, 'Reg': 4},
    'LandContour': {'Low': 1, 'HLS': 2, 'Bnk': 3, 'Lvl': 4},
    'ExterQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'ExterCond': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtQual': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtCond': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtExposure': {'NA': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'NA': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'HeatingQC': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'Functional': {'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8},
    'FireplaceQu': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
    'GarageQual': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'GarageCond': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'PavedDrive': {'N': 1, 'P': 2, 'Y': 3},
    'PoolQC': {'NA': 0, 'Fa': 1, 'TA': 2, 'Gd': 3, 'Ex': 4},
    'Fence': {'NA': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
}

# Nominal Features to be one-hot encoded
nominal_features = [
    'MSZoning', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2',
    'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
    'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir',
    'Electrical', 'GarageType', 'SaleType', 'SaleCondition'
]

# Step 8 - Apply ordinal / onehot on categorical columns on train and test data
def apply_ordinal_encoding(df: pd.DataFrame, ordinal_map: dict) -> pd.DataFrame:
    df_copy = df.copy()
    for col, mapping in ordinal_map.items():
        # Fill NA with 'NA' if it's in mapping
        if 'NA' in mapping:
            df_copy[col] = df_copy[col].fillna('NA')
        else:
            df_copy[col] = df_copy[col].fillna(df_copy[col].mode()[0])
        df_copy[col] = df_copy[col].map(mapping)
    return df_copy

def apply_one_hot_encoding(df: pd.DataFrame, nominal_cols: list) -> pd.DataFrame:
    return pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# Apply ordinal encoding first
train_df_encoded = apply_ordinal_encoding(train_df, ordinal_features)
test_df_encoded = apply_ordinal_encoding(test_df, ordinal_features)

# Then one-hot encode nominal columns
train_df_encoded = apply_one_hot_encoding(train_df_encoded, nominal_features)
test_df_encoded = apply_one_hot_encoding(test_df_encoded, nominal_features)

# Ensure test and train have the same columns
train_df_encoded, test_df_encoded = train_df_encoded.align(test_df_encoded, join='left', axis=1, fill_value=0)

# Convert entire DataFrame to numeric dtype (inplace conversion)
train_df_encoded = train_df_encoded.apply(pd.to_numeric, errors='coerce')
test_df_encoded = test_df_encoded.apply(pd.to_numeric, errors='coerce')

# Step 9 - Create Features list by excluding SalePrice, Id columns and target as SalePrice
# Define features and target
X = train_df_encoded.drop(['SalePrice', 'Id'], axis=1, errors='ignore')
y = train_df_encoded['SalePrice']

# For prediction output
test_ids = test_df_encoded['Id']
X_test = test_df_encoded.drop(['SalePrice', 'Id'], axis=1, errors='ignore')  # SalePrice may not exist in test


# Step 10 - Find RMSE using LinearRegression using cross val score
lin_reg = LinearRegression()
# Log-transform the target variable
y_log = np.log1p(y)
# Cross-validate with negative RMSE scoring on log scale
scores = cross_val_score(lin_reg, X, y_log, scoring='neg_root_mean_squared_error', cv=5)
print("==================================================")
print(f"Linear Regression CV RMSE on log scale: {-np.mean(scores):.4f}")
print("==================================================")

# Step 11 - Perform predictions on log transform and save the values to submission.csv file
# 1. Fit model on full training data (log-transformed SalePrice)
lin_reg.fit(X, np.log1p(y))  # log1p for better modeling of skewed target
# 2. Predict on test data (log scale)
log_preds = lin_reg.predict(X_test)
# 3. Convert predictions back to original scale
preds = np.expm1(log_preds)

# 4. Create submission DataFrame
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds
})

# 5. Save to CSV (no index)
# submission.to_csv(r"D:\Career-Related\projects\learning\kaggle-dataset-examples\home-data-for-ml-course\submission.csv", index=False)
# print("✅ Submission file 'submission.csv' created successfully.")






=========== Check Duplicates in DataFrame =========== Start
Duplicates does not exist. Go to next step for checking nulls
Train Data Duplicate Exists: False
Duplicates does not exist. Go to next step for checking nulls
Test Data Duplicate Exists: False
=========== Check Duplicates in DataFrame =========== End
=========== Check Nulls in DataFrame =========== Start
Null Values exists for DataFrame. Need to replace nulls appropriately
Train Data Null Values Exists: True
Null Values exists for DataFrame. Need to replace nulls appropriately
Test Data Null Values Exists: True
=========== Check Nulls in DataFrame =========== End
=========== Get Nulls Columns in DataFrame =========== Start
Train Data Null Values Exists: ['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']
Test Data Null Va

#### Data Cleanup

In [63]:
# Check Duplicates
print("=========== Check Duplicates in DataFrame =========== Start")
train_data_duplicate_exists = check_duplicates(train_df)
print("Train Data Duplicate Exists: "+str(train_data_duplicate_exists))
test_data_duplicate_exists = check_duplicates(test_df)
print("Test Data Duplicate Exists: "+str(test_data_duplicate_exists))
print("=========== Check Duplicates in DataFrame =========== End")

print("=========== Check Nulls in DataFrame =========== Start")
train_data_null_values_exists = check_null_values(train_df)
print("Train Data Null Values Exists: "+str(train_data_null_values_exists))
test_data_null_values_exists = check_null_values(test_df)
print("Test Data Null Values Exists: "+str(test_data_null_values_exists))
print("=========== Check Nulls in DataFrame =========== End")

print("=========== Get Nulls Columns in DataFrame =========== Start")
train_data_null_cols = get_null_cols_list(train_df)
print("Train Data Null Values Exists: "+str(train_data_null_cols))
test_data_null_cols = get_null_cols_list(test_df)
print("Test Data Null Values Exists: "+str(test_data_null_cols))
print("=========== Get Nulls Columns in DataFrame =========== End")

print("=========== Update Nulls Columns in DataFrame =========== Start")
# On train data
fill_values = get_fill_values(train_df)
apply_fill_values(train_df, fill_values)

# On test data — use same values
apply_fill_values(test_df, fill_values)
print("=========== Update Nulls Columns in DataFrame =========== End")

print("=========== Check Nulls in DataFrame =========== Start")
train_data_null_values_exists = check_null_values(train_df)
print("Train Data Null Values Exists: "+str(train_data_null_values_exists))
test_data_null_values_exists = check_null_values(test_df)
print("Test Data Null Values Exists: "+str(test_data_null_values_exists))
print("=========== Check Nulls in DataFrame =========== End")
# Problem Statement: Identify features from train data which affect SalePrice
# Approach:
#       1. Identify all numeric columns in train data  
#       2. Find correlation of all the columns against SalePrice and identify the list with highest value > 0.5 and < 0.5
#       3. Store the columns in a numerical features list
#       4. Identify categorical columns in train data
#       5. Perform ANOVA test on categorical columns of train data and identify columns with low p-val 
#       6. Seggregate the categorical columns into Ordinal and Nominal for encoding
#       6. Convert the categorical columns to label encoding
#       7. Append to numerical features list created during numerical analysis
#       8.

# Step1 - Identify all numeric columns in train data
correlation_cols = train_df.corr(numeric_only=True).corr()['SalePrice'].sort_values(ascending=False)
# Step 2 - Find correlation of all the columns against SalePrice and identify the list with highest value > 0.5 and < 0.5
strong_corr_cols = correlation_cols[abs(correlation_cols) > 0.5].drop('SalePrice')
# Step 3 - Store the columns in a features list
numerical_features_list = strong_corr_cols.index.tolist()
# Step 4 - Identify categorical columns in train data
categorical_cols = train_df.select_dtypes(include='object').columns
print(categorical_cols)

categorical_features_list = []
for col in categorical_cols:
    groups = [group['SalePrice'].values for _, group in train_df.groupby(col)]
    f_stat, p_val = f_oneway(*groups)
    if p_val < 0.05:
        categorical_features_list.append(col)
    else:
        print(f"{col} is not candidate for encoding. Adding to categorical_features_list")
print(len(categorical_features_list), len(categorical_cols))


=========== Check Duplicates in DataFrame =========== Start
Duplicates does not exist. Go to next step for checking nulls
Train Data Duplicate Exists: False
Duplicates does not exist. Go to next step for checking nulls
Test Data Duplicate Exists: False
=========== Check Duplicates in DataFrame =========== End
=========== Check Nulls in DataFrame =========== Start
Null Values does not exist for DataFrame.
Train Data Null Values Exists: False
Null Values does not exist for DataFrame.
Test Data Null Values Exists: False
=========== Check Nulls in DataFrame =========== End
=========== Get Nulls Columns in DataFrame =========== Start
Train Data Null Values Exists: []
Test Data Null Values Exists: []
=========== Get Nulls Columns in DataFrame =========== End
=========== Update Nulls Columns in DataFrame =========== Start
Missing values replaced using training data statistics.
Missing values replaced using training data statistics.
=========== Update Nulls Columns in DataFrame =========== End

### Data Statistics on train data

#### Field Descriptions
    
Here's a brief version of what you'll find in the data description file.

    SalePrice - the property's sale price in dollars. This is the target variable that you're trying to predict.
    MSSubClass: The building class
    MSZoning: The general zoning classification
    LotFrontage: Linear feet of street connected to property
    LotArea: Lot size in square feet
    Street: Type of road access
    Alley: Type of alley access
    LotShape: General shape of property
    LandContour: Flatness of the property
    Utilities: Type of utilities available
    LotConfig: Lot configuration
    LandSlope: Slope of property
    Neighborhood: Physical locations within Ames city limits
    Condition1: Proximity to main road or railroad
    Condition2: Proximity to main road or railroad (if a second is present)
    BldgType: Type of dwelling
    HouseStyle: Style of dwelling
    OverallQual: Overall material and finish quality
    OverallCond: Overall condition rating
    YearBuilt: Original construction date
    YearRemodAdd: Remodel date
    RoofStyle: Type of roof
    RoofMatl: Roof material
    Exterior1st: Exterior covering on house
    Exterior2nd: Exterior covering on house (if more than one material)
    MasVnrType: Masonry veneer type
    MasVnrArea: Masonry veneer area in square feet
    ExterQual: Exterior material quality
    ExterCond: Present condition of the material on the exterior
    Foundation: Type of foundation
    BsmtQual: Height of the basement
    BsmtCond: General condition of the basement
    BsmtExposure: Walkout or garden level basement walls
    BsmtFinType1: Quality of basement finished area
    BsmtFinSF1: Type 1 finished square feet
    BsmtFinType2: Quality of second finished area (if present)
    BsmtFinSF2: Type 2 finished square feet
    BsmtUnfSF: Unfinished square feet of basement area
    TotalBsmtSF: Total square feet of basement area
    Heating: Type of heating
    HeatingQC: Heating quality and condition
    CentralAir: Central air conditioning
    Electrical: Electrical system
    1stFlrSF: First Floor square feet
    2ndFlrSF: Second floor square feet
    LowQualFinSF: Low quality finished square feet (all floors)
    GrLivArea: Above grade (ground) living area square feet
    BsmtFullBath: Basement full bathrooms
    BsmtHalfBath: Basement half bathrooms
    FullBath: Full bathrooms above grade
    HalfBath: Half baths above grade
    Bedroom: Number of bedrooms above basement level
    Kitchen: Number of kitchens
    KitchenQual: Kitchen quality
    TotRmsAbvGrd: Total rooms above grade (does not include bathrooms)
    Functional: Home functionality rating
    Fireplaces: Number of fireplaces
    FireplaceQu: Fireplace quality
    GarageType: Garage location
    GarageYrBlt: Year garage was built
    GarageFinish: Interior finish of the garage
    GarageCars: Size of garage in car capacity
    GarageArea: Size of garage in square feet
    GarageQual: Garage quality
    GarageCond: Garage condition
    PavedDrive: Paved driveway
    WoodDeckSF: Wood deck area in square feet
    OpenPorchSF: Open porch area in square feet
    EnclosedPorch: Enclosed porch area in square feet
    3SsnPorch: Three season porch area in square feet
    ScreenPorch: Screen porch area in square feet
    PoolArea: Pool area in square feet
    PoolQC: Pool quality
    Fence: Fence quality
    MiscFeature: Miscellaneous feature not covered in other categories
    MiscVal: $Value of miscellaneous feature
    MoSold: Month Sold
    YrSold: Year Sold
    SaleType: Type of sale
    SaleCondition: Condition of sale

#### Goal : 
    Problem Statement: Identify features from train data which affect SalePrice
    Approach:
        1. Identify all numeric columns in train data  
        2. Find correlation of all the columns against SalePrice and identify the list with highest value > 0.5 and < 0.5
        3. Store the columns in a numerical features list
        4. Identify categorical columns in train data
        5. Perform ANOVA test on categorical columns of train data and identify columns with 
        6. Convert the categorical columns to label encoding
        7. Append to numerical features list created during numerical analysis
        8. 
        
  

In [59]:
# Step1 - Identify all numeric columns in train data
correlation_cols = train_df.corr(numeric_only=True).corr()['SalePrice'].sort_values(ascending=False)
# Step 2 - Find correlation of all the columns against SalePrice and identify the list with highest value > 0.5 and < 0.5
strong_corr_cols = correlation_cols[abs(correlation_cols) > 0.5].drop('SalePrice')
# Step 3 - Store the columns in a features list
numerical_features_list = strong_corr_cols.index.tolist()
# Step 4 - Identify categorical columns in train data
categorical_cols = train_df.select_dtypes(include='object').columns
print(categorical_cols)

categorical_features_list = []
for col in categorical_cols:
    groups = [group['SalePrice'].values for _, group in train_df.groupby(col)]
    f_stat, p_val = f_oneway(*groups)
    if p_val < 0.05:
        # print(f"{col} is candidate for encoding. Adding to categorical_features_list")
        print(f"F-statistic: {f_stat}, p-value: {p_val}")
        categorical_features_list.append(col)
print(categorical_features_list)
    


Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object')
F-statistic: 43.84028167245718, p-value: 8.817633866272648e-35
F-statistic: 40.132851662262944, p-value: 6.447523852011766e-25
F-statistic: 12.850188333283924, p-value: 2.7422167521379096e-08
F-statistic: 7.80995412346779, p-value: 3.163167473604189e-06
F-statistic: 71.78486512058278, p-value: 1.5586002827707996e-225
F-statistic: 6.1

In [60]:
for col in categorical_cols:
    print(f"{col}: {train_df[col].unique()}")

MSZoning: ['RL' 'RM' 'C (all)' 'FV' 'RH']
Street: ['Pave' 'Grvl']
Alley: ['Grvl' 'Pave']
LotShape: ['Reg' 'IR1' 'IR2' 'IR3']
LandContour: ['Lvl' 'Bnk' 'Low' 'HLS']
Utilities: ['AllPub' 'NoSeWa']
LotConfig: ['Inside' 'FR2' 'Corner' 'CulDSac' 'FR3']
LandSlope: ['Gtl' 'Mod' 'Sev']
Neighborhood: ['CollgCr' 'Veenker' 'Crawfor' 'NoRidge' 'Mitchel' 'Somerst' 'NWAmes'
 'OldTown' 'BrkSide' 'Sawyer' 'NridgHt' 'NAmes' 'SawyerW' 'IDOTRR'
 'MeadowV' 'Edwards' 'Timber' 'Gilbert' 'StoneBr' 'ClearCr' 'NPkVill'
 'Blmngtn' 'BrDale' 'SWISU' 'Blueste']
Condition1: ['Norm' 'Feedr' 'PosN' 'Artery' 'RRAe' 'RRNn' 'RRAn' 'PosA' 'RRNe']
Condition2: ['Norm' 'Artery' 'RRNn' 'Feedr' 'PosN' 'PosA' 'RRAn' 'RRAe']
BldgType: ['1Fam' '2fmCon' 'Duplex' 'TwnhsE' 'Twnhs']
HouseStyle: ['2Story' '1Story' '1.5Fin' '1.5Unf' 'SFoyer' 'SLvl' '2.5Unf' '2.5Fin']
RoofStyle: ['Gable' 'Hip' 'Gambrel' 'Mansard' 'Flat' 'Shed']
RoofMatl: ['CompShg' 'WdShngl' 'Metal' 'WdShake' 'Membran' 'Tar&Grv' 'Roll'
 'ClyTile']
Exterior1st: ['VinylS